# Aura Score Analyzer

Website: [AuraAnalyzer.pro](https://auraanalyzer.pro)

## Overview
This Google Colab notebook powers the Aura Score Analyzer project. It loads several TensorFlow image models, analyzes one uploaded photo, and returns a final Aura score, grade, explanation report, and JSON output.

## Technologies Used
- Python
- Google Colab
- TensorFlow / Keras
- `tf_keras` for legacy `.h5` compatibility
- TensorFlow.js conversion tooling
- Gradio
- OpenCV
- Pillow
- NumPy
- Matplotlib
- GitHub


In [ ]:
# Cell 1: Install packages
# TensorFlow is preinstalled in Google Colab. These packages support image I/O, face detection, display, legacy Keras .h5 loading, TensorFlow.js conversion, and the website interface.
%pip install -q opencv-python-headless pillow matplotlib tf_keras tensorflowjs gradio


In [ ]:
# Cell 2: Download project files directly from GitHub
# After you upload this folder to GitHub, Colab will clone the repo and get the models and Data folder automatically.
import os
import shutil
import subprocess
from pathlib import Path

GITHUB_REPO_URL = 'https://github.com/mhirez/Aura-Score-Analyzer'
GITHUB_BRANCH = 'main'
PROJECT_DIR = '/content/Aura-Score-Analyzer'

def normalize_github_clone_url(repo_url):
    repo_url = repo_url.strip().rstrip('/')
    if repo_url.endswith('.git'):
        return repo_url
    return repo_url + '.git'

def run_command(command):
    print('Running:', ' '.join(command))
    completed = subprocess.run(command, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(f'Command failed: {" ".join(command)}')
    return completed

def count_files(folder_path):
    folder = Path(folder_path)
    if not folder.exists():
        return 0
    return sum(1 for path in folder.rglob('*') if path.is_file())

clone_url = normalize_github_clone_url(GITHUB_REPO_URL)

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

try:
    run_command(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, clone_url, PROJECT_DIR])
except Exception as branch_error:
    print('Branch-specific clone failed. Trying the repository default branch instead.')
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    run_command(['git', 'clone', '--depth', '1', clone_url, PROJECT_DIR])

DATA_DIR = os.path.join(PROJECT_DIR, 'Data')
FEATURE_MODELS_DIR = os.path.join(PROJECT_DIR, 'Feature Detection Models')

print('\nProject files are ready.')
print('Project directory:', PROJECT_DIR)
print('Data directory:', DATA_DIR)
print('Feature models directory:', FEATURE_MODELS_DIR)
print('Dataset file count:', count_files(DATA_DIR))
print('Feature model file count:', count_files(FEATURE_MODELS_DIR))
print('\nNext: run Cell 3 to extract the model zip files.')


Running: git clone --depth 1 --branch main https://github.com/mhirez/Aura-Score-Analyzer.git /content/Aura-Score-Analyzer

Project files are ready.
Project directory: /content/Aura-Score-Analyzer
Data directory: /content/Aura-Score-Analyzer/Data
Feature models directory: /content/Aura-Score-Analyzer/Feature Detection Models
Dataset file count: 5751
Feature model file count: 3

Next: run Cell 3 to extract the model zip files.


In [ ]:
# Cell 3: Extract model zip files from the GitHub project folder
import os
import shutil
import zipfile
from pathlib import Path

CLOSED_ARMS_ZIP_NAME = 'crossed_open_arms.zip'
SERIOUS_FACE_ZIP_NAME = 'smile_not_smile_model.zip'
GLASSES_ZIP_NAME = 'glasses_no_glasses.zip'
GENERAL_ZIP_NAME = 'general_model.zip'

MODEL_EXTRACT_ROOT = '/content/aura_models'
CROSSED_MODEL_DIR = os.path.join(MODEL_EXTRACT_ROOT, 'closed_arms_model')
SMILE_MODEL_DIR = os.path.join(MODEL_EXTRACT_ROOT, 'serious_face_model')
GLASSES_MODEL_DIR = os.path.join(MODEL_EXTRACT_ROOT, 'glasses_model')
GENERAL_MODEL_DIR = os.path.join(MODEL_EXTRACT_ROOT, 'general_model')

def find_project_file(*relative_paths):
    for relative_path in relative_paths:
        candidate = os.path.join(PROJECT_DIR, relative_path)
        if os.path.exists(candidate):
            return candidate
    tried_paths = [os.path.join(PROJECT_DIR, path) for path in relative_paths]
    raise FileNotFoundError('Could not find project file. Tried: ' + ', '.join(tried_paths))

def extract_zip(zip_path, output_dir):
    zip_path = os.path.abspath(zip_path)
    output_dir = os.path.abspath(output_dir)

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in zip_ref.infolist():
            member_path = os.path.abspath(os.path.join(output_dir, member.filename))
            if not (member_path == output_dir or member_path.startswith(output_dir + os.sep)):
                raise ValueError(f'Unsafe path found inside zip file: {member.filename}')
        zip_ref.extractall(output_dir)

    return output_dir

CROSSED_ZIP_PATH = find_project_file(
    os.path.join('Feature Detection Models', CLOSED_ARMS_ZIP_NAME),
    CLOSED_ARMS_ZIP_NAME
)
SMILE_ZIP_PATH = find_project_file(
    os.path.join('Feature Detection Models', SERIOUS_FACE_ZIP_NAME),
    SERIOUS_FACE_ZIP_NAME
)
GLASSES_ZIP_PATH = find_project_file(
    os.path.join('Feature Detection Models', GLASSES_ZIP_NAME),
    GLASSES_ZIP_NAME
)
GENERAL_ZIP_PATH = find_project_file(
    GENERAL_ZIP_NAME,
    os.path.join('Feature Detection Models', GENERAL_ZIP_NAME)
)

extract_zip(CROSSED_ZIP_PATH, CROSSED_MODEL_DIR)
extract_zip(SMILE_ZIP_PATH, SMILE_MODEL_DIR)
extract_zip(GLASSES_ZIP_PATH, GLASSES_MODEL_DIR)
extract_zip(GENERAL_ZIP_PATH, GENERAL_MODEL_DIR)

print('Model files were extracted to:')
print('Closed arms model:', CROSSED_MODEL_DIR)
print('Serious face model:', SMILE_MODEL_DIR)
print('Glasses model:', GLASSES_MODEL_DIR)
print('General aura model:', GENERAL_MODEL_DIR)
print('\nThe project Data folder is also available at:', DATA_DIR)
print('Next: run Cells 4, 5, and 6 once to prepare and load the models. Then rerun Cell 7 for each new image, or run Cell 8 for the website interface.')


Model files were extracted to:
Closed arms model: /content/aura_models/closed_arms_model
Serious face model: /content/aura_models/serious_face_model
Glasses model: /content/aura_models/glasses_model
General aura model: /content/aura_models/general_model

The project Data folder is also available at: /content/Aura-Score-Analyzer/Data
Next: run Cells 4, 5, and 6 once to prepare and load the models. Then rerun Cell 7 for each new image, or run Cell 8 for the website interface.


In [ ]:
# Cell 4: Face crop and display helper functions
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

FACE_CROP_PATH = '/content/face_crop.png'

def crop_face(image_path, output_path=FACE_CROP_PATH, margin_ratio=0.35):
    pil_image = Image.open(image_path).convert('RGB')
    rgb_image = np.array(pil_image)

    cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    face_cascade = cv2.CascadeClassifier(cascade_path)

    if face_cascade.empty():
        pil_image.save(output_path)
        message = 'Warning: Haar Cascade could not be loaded. The full image will be used for serious face and glasses models.'
        return output_path, False, message

    gray_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(
        gray_image,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )

    if len(faces) == 0:
        pil_image.save(output_path)
        message = 'Warning: no face was detected. The full image will be used for serious face and glasses models.'
        return output_path, False, message

    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
    margin_x = int(w * margin_ratio)
    margin_y = int(h * margin_ratio)

    image_height, image_width = rgb_image.shape[:2]
    x1 = max(0, x - margin_x)
    y1 = max(0, y - margin_y)
    x2 = min(image_width, x + w + margin_x)
    y2 = min(image_height, y + h + margin_y)

    face_crop = rgb_image[y1:y2, x1:x2]
    Image.fromarray(face_crop).save(output_path)
    message = 'Face crop succeeded. The face crop will be used for serious face and glasses models.'
    return output_path, True, message

def display_original_and_face_crop(input_image_path, face_crop_path, face_crop_used):
    original_image = Image.open(input_image_path).convert('RGB')
    crop_or_fallback_image = Image.open(face_crop_path).convert('RGB')

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(original_image)
    plt.title('Original image')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(crop_or_fallback_image)
    plt.title('Face crop' if face_crop_used else 'Full image fallback')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

print('Face crop helper functions are ready. The actual photo will be uploaded and cropped in Cell 7.')


Face crop helper functions are ready. The actual photo will be uploaded and cropped in Cell 7.


In [ ]:
# Cell 5: Helper functions for model loading, labels, preprocessing, probabilities, and scoring
import json
import math
import os
import re
from pathlib import Path

import numpy as np

# Old Teachable Machine .h5 exports often load more reliably with legacy Keras than with Keras 3.
# This environment variable must be set before importing TensorFlow in a fresh runtime.
os.environ.setdefault('TF_USE_LEGACY_KERAS', '1')

import tensorflow as tf

try:
    import tf_keras
except Exception:
    tf_keras = None
from PIL import Image, ImageOps

NORMALIZATION_MODE = 'teachable_machine'
DEFAULT_INPUT_SIZE = (224, 224)
EQUAL_MODEL_WEIGHT = 33.3333
AURA_EQUATION = 'final_aura_score = 100 * ((P(closed_arms) + P(serious_face) + P(glasses)) / 3)'

try:
    RESAMPLE_FILTER = Image.Resampling.LANCZOS
except AttributeError:
    RESAMPLE_FILTER = Image.LANCZOS

class CompatibleDepthwiseConv2D(tf.keras.layers.DepthwiseConv2D):
    @classmethod
    def from_config(cls, config):
        config = dict(config)
        config.pop('groups', None)
        return super().from_config(config)

class SavedModelPredictor:
    def __init__(self, saved_model_dir):
        self.saved_model_dir = saved_model_dir
        self.loaded = tf.saved_model.load(saved_model_dir)

        if 'serving_default' in self.loaded.signatures:
            self.signature = self.loaded.signatures['serving_default']
        else:
            signature_keys = list(self.loaded.signatures.keys())
            if not signature_keys:
                raise ValueError('SavedModel has no callable signatures.')
            self.signature = self.loaded.signatures[signature_keys[0]]

        positional_inputs, keyword_inputs = self.signature.structured_input_signature
        if keyword_inputs:
            self.input_name, self.input_spec = next(iter(keyword_inputs.items()))
            self.uses_keyword_input = True
        elif positional_inputs:
            self.input_name = None
            self.input_spec = positional_inputs[0]
            self.uses_keyword_input = False
        else:
            raise ValueError('SavedModel signature does not expose an input tensor.')

        try:
            self.input_shape = tuple(self.input_spec.shape.as_list())
        except Exception:
            self.input_shape = None

    def predict(self, batch, verbose=0):
        input_tensor = tf.convert_to_tensor(batch, dtype=tf.float32)
        if self.uses_keyword_input:
            outputs = self.signature(**{self.input_name: input_tensor})
        else:
            outputs = self.signature(input_tensor)

        if isinstance(outputs, dict):
            output_tensor = next(iter(outputs.values()))
        else:
            output_tensor = outputs
        return output_tensor.numpy()

def find_model_file(model_dir):
    model_dir = Path(model_dir)
    if not model_dir.exists():
        raise FileNotFoundError(f'Model directory does not exist: {model_dir}')

    search_patterns = [
        '**/*.keras',
        '**/keras_model.h5',
        '**/model.h5',
        '**/*.h5'
    ]
    for pattern in search_patterns:
        matches = sorted(model_dir.glob(pattern), key=lambda p: (len(str(p)), str(p).lower()))
        if matches:
            return str(matches[0])

    saved_model_dirs = sorted({path.parent for path in model_dir.rglob('saved_model.pb')}, key=lambda p: (len(str(p)), str(p).lower()))
    if saved_model_dirs:
        return str(saved_model_dirs[0])

    model_json_files = sorted(model_dir.rglob('model.json'), key=lambda p: (len(str(p)), str(p).lower()))
    if model_json_files:
        return str(model_json_files[0])

    raise FileNotFoundError(f'No supported model file was found inside: {model_dir}')

def clean_label(label_text):
    label_text = str(label_text).strip()
    label_text = re.sub(r'^\s*\d+\s*[:.)-]?\s*', '', label_text)
    return label_text.strip()

def read_labels(model_dir):
    model_dir = Path(model_dir)
    label_files = sorted(model_dir.rglob('labels.txt'), key=lambda p: (len(str(p)), str(p).lower()))
    for label_file in label_files:
        labels = []
        with open(label_file, 'r', encoding='utf-8') as file:
            for line in file:
                cleaned = clean_label(line)
                if cleaned:
                    labels.append(cleaned)
        if labels:
            return labels

    metadata_files = sorted(model_dir.rglob('metadata.json'), key=lambda p: (len(str(p)), str(p).lower()))
    for metadata_file in metadata_files:
        with open(metadata_file, 'r', encoding='utf-8') as file:
            metadata = json.load(file)
        raw_labels = metadata.get('labels') or metadata.get('modelLabels') or metadata.get('classes')
        if isinstance(raw_labels, list):
            labels = []
            for item in raw_labels:
                if isinstance(item, dict):
                    value = item.get('name') or item.get('label') or item.get('displayName')
                else:
                    value = item
                cleaned = clean_label(value)
                if cleaned:
                    labels.append(cleaned)
            if labels:
                return labels

    return []

def load_keras_model(model_path):
    model_path = str(model_path)
    first_error = None
    load_errors = []

    is_h5_model = model_path.lower().endswith(('.h5', '.hdf5'))
    is_tfjs_model = model_path.lower().endswith('model.json')

    if is_h5_model and tf_keras is not None:
        try:
            return tf_keras.models.load_model(model_path, compile=False)
        except Exception as error:
            first_error = error
            load_errors.append(f'tf_keras legacy loader failed: {error}')

    if not is_tfjs_model:
        try:
            return tf.keras.models.load_model(model_path, compile=False)
        except Exception as error:
            first_error = first_error or error
            load_errors.append(f'tf.keras loader failed: {error}')

    error_message = '\n'.join(load_errors) if load_errors else str(first_error)
    if 'DepthwiseConv2D' in error_message and 'groups' in error_message:
        try:
            return tf.keras.models.load_model(
                model_path,
                compile=False,
                custom_objects={'DepthwiseConv2D': CompatibleDepthwiseConv2D}
            )
        except Exception as depthwise_error:
            first_error = first_error or depthwise_error
            load_errors.append(f'DepthwiseConv2D compatibility loader failed: {depthwise_error}')

        if tf_keras is not None:
            try:
                return tf_keras.models.load_model(
                    model_path,
                    compile=False,
                    custom_objects={'DepthwiseConv2D': CompatibleDepthwiseConv2D}
                )
            except Exception as depthwise_legacy_error:
                first_error = first_error or depthwise_legacy_error
                load_errors.append(f'Legacy DepthwiseConv2D compatibility loader failed: {depthwise_legacy_error}')

    if os.path.isdir(model_path) or os.path.exists(os.path.join(model_path, 'saved_model.pb')):
        try:
            return SavedModelPredictor(model_path)
        except Exception as saved_model_error:
            first_error = first_error or saved_model_error
            load_errors.append(f'SavedModel loader failed: {saved_model_error}')

    if model_path.lower().endswith('model.json'):
        try:
            import tensorflowjs as tfjs
            return tfjs.converters.load_keras_model(model_path)
        except Exception as tfjs_error:
            load_errors.append(f'TensorFlow.js converter failed: {tfjs_error}')
            raise RuntimeError(
                'A TensorFlow.js model.json was found, but it could not be loaded by the optional tensorflowjs converter. '
                'Keras .h5, .keras, or SavedModel exports are preferred for this notebook. '
                f'Original error: {tfjs_error}'
            ) from tfjs_error

    details = '\n\n'.join(load_errors) if load_errors else str(first_error)
    raise RuntimeError(f'Could not load model at {model_path}. Loader details:\n{details}') from first_error

def get_input_size(model):
    shape = getattr(model, 'input_shape', None)
    if isinstance(shape, list) and shape:
        shape = shape[0]

    if shape is None and hasattr(model, 'inputs') and model.inputs:
        shape = model.inputs[0].shape

    if hasattr(shape, 'as_list'):
        shape = shape.as_list()

    try:
        dims = [None if dim is None else int(dim) for dim in list(shape)]
    except Exception:
        return DEFAULT_INPUT_SIZE

    height = None
    width = None
    if len(dims) == 4:
        if dims[-1] in (1, 3, 4):
            height, width = dims[1], dims[2]
        elif dims[1] in (1, 3, 4):
            height, width = dims[2], dims[3]
    elif len(dims) == 3:
        if dims[-1] in (1, 3, 4):
            height, width = dims[0], dims[1]
        elif dims[0] in (1, 3, 4):
            height, width = dims[1], dims[2]

    if height is None or width is None or height <= 0 or width <= 0:
        return DEFAULT_INPUT_SIZE

    return int(height), int(width)

def preprocess_image(image_path, model):
    height, width = get_input_size(model)
    image = Image.open(image_path).convert('RGB')
    image = ImageOps.fit(image, (width, height), RESAMPLE_FILTER)
    image_array = np.asarray(image).astype(np.float32)

    if NORMALIZATION_MODE == 'teachable_machine':
        image_array = (image_array / 127.5) - 1.0
    elif NORMALIZATION_MODE == 'zero_to_one':
        image_array = image_array / 255.0
    else:
        raise ValueError('NORMALIZATION_MODE must be teachable_machine or zero_to_one.')

    return np.expand_dims(image_array, axis=0)

def sigmoid(value):
    return 1.0 / (1.0 + math.exp(-float(value)))

def convert_to_probability_vector(raw_values, labels):
    values = np.asarray(raw_values, dtype=np.float32).reshape(-1)
    if values.size == 0:
        raise ValueError('Model returned an empty prediction.')

    if values.size == 1 and len(labels) == 2:
        p = float(values[0])
        if p < 0.0 or p > 1.0:
            p = sigmoid(p)
        p = float(np.clip(p, 0.0, 1.0))
        return np.asarray([1.0 - p, p], dtype=np.float32)

    if values.size == 1:
        p = float(values[0])
        if p < 0.0 or p > 1.0:
            p = sigmoid(p)
        p = float(np.clip(p, 0.0, 1.0))
        return np.asarray([p], dtype=np.float32)

    values = values.astype(np.float64)
    values_sum = float(np.sum(values))
    values_are_probabilities = np.all(values >= 0.0) and np.all(values <= 1.0) and np.isclose(values_sum, 1.0, atol=1e-3)

    if values_are_probabilities:
        probabilities = values
    else:
        shifted_values = values - np.max(values)
        exp_values = np.exp(shifted_values)
        probabilities = exp_values / np.sum(exp_values)

    return probabilities.astype(np.float32)

def align_labels(labels, probability_count):
    aligned = [clean_label(label) for label in labels if clean_label(label)]
    if len(aligned) < probability_count:
        start = len(aligned)
        aligned.extend([f'class_{index}' for index in range(start, probability_count)])
    elif len(aligned) > probability_count:
        aligned = aligned[:probability_count]
    return aligned

def predict_model(model, image_path, labels):
    batch = preprocess_image(image_path, model)
    raw_prediction = model.predict(batch, verbose=0)

    if isinstance(raw_prediction, (list, tuple)):
        raw_prediction = raw_prediction[0]

    raw_array = np.asarray(raw_prediction)
    if raw_array.ndim > 1:
        raw_values = raw_array[0]
    else:
        raw_values = raw_array

    probabilities = convert_to_probability_vector(raw_values, labels)
    aligned_labels = align_labels(labels, len(probabilities))
    probability_map = {aligned_labels[index]: float(probabilities[index]) for index in range(len(probabilities))}

    return {
        'labels': aligned_labels,
        'raw_values': [float(value) for value in np.asarray(raw_values).reshape(-1)],
        'probabilities': [float(value) for value in probabilities],
        'probability_map': probability_map
    }

def normalize_label_text(text):
    text = str(text).strip().lower()
    text = re.sub(r'^\s*\d+\s*[:.)-]?\s*', '', text)
    text = text.replace('_', ' ').replace('-', ' ')
    text = re.sub(r'[^a-z0-9 ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def find_target_label(labels, possible_names):
    normalized_labels = [normalize_label_text(label) for label in labels]
    normalized_targets = [normalize_label_text(name) for name in possible_names]

    for index, label in enumerate(normalized_labels):
        if label in normalized_targets:
            return index, labels[index]

    for index, label in enumerate(normalized_labels):
        for target in sorted(normalized_targets, key=len, reverse=True):
            if not target:
                continue
            target_word_count = len(target.split())
            if target_word_count >= 2 and target in label:
                return index, labels[index]
            if target_word_count == 1 and label.startswith(target + ' '):
                return index, labels[index]

    return None, None

def get_target_probability(prediction, target_names, negative_names=None):
    labels = prediction['labels']
    probabilities = prediction['probabilities']

    target_index, target_label = find_target_label(labels, target_names)
    if target_index is not None:
        return float(probabilities[target_index]), target_label, 'direct target label match'

    if negative_names:
        negative_index, negative_label = find_target_label(labels, negative_names)
        if negative_index is not None and len(probabilities) == 2:
            return float(1.0 - probabilities[negative_index]), f'not {negative_label}', 'binary complement from negative label'

    raise ValueError(
        'Could not identify the target class from labels. '
        f'Labels found: {labels}. Target names tried: {target_names}'
    )

def grade_from_score(score):
    if score >= 80.0:
        return 'Very strong aura'
    if score >= 60.0:
        return 'Strong aura'
    if score >= 40.0:
        return 'Medium aura'
    if score >= 20.0:
        return 'Low aura'
    return 'Very low aura'

def build_reason(reason_subject, probability):
    probability = float(probability)
    if probability >= 0.80:
        return f'The model strongly detected {reason_subject}, so this increased the aura score.'
    if probability >= 0.60:
        return f'The model detected mostly {reason_subject}, so this moderately increased the aura score.'
    if probability >= 0.40:
        return f'The model was uncertain about {reason_subject}, so this gave a medium contribution to the aura score.'
    if probability >= 0.20:
        return f'The model weakly detected {reason_subject}, so this only slightly increased the aura score.'
    return f'The model did not strongly detect {reason_subject}, so this added very little to the aura score.'

def friendly_label_name(label):
    label = str(label)
    replacements = {
        'crossed_arms': 'closed_arms',
        'crossed arms': 'closed arms',
        'crossed_hands': 'closed_hands',
        'crossed hands': 'closed hands',
        'crossed': 'closed',
        'non_smile': 'serious_face',
        'not_smile': 'serious_face',
        'not smile': 'serious face',
        'no_smile': 'serious_face',
        'not smiling': 'serious face',
        'neutral': 'serious_face',
        'smile': 'not_serious_face',
        'smiling': 'not_serious_face'
    }
    normalized = label.lower()
    return replacements.get(normalized, label)

def compare_general_model_to_program(final_score, general_model_result):
    if not general_model_result:
        return None

    transparent_probability_percent = float(final_score)
    general_probability_percent = float(general_model_result['target_probability'] * 100.0)
    signed_difference = general_probability_percent - transparent_probability_percent
    absolute_difference = abs(signed_difference)

    if absolute_difference <= 10.0:
        agreement_level = 'Close agreement'
    elif absolute_difference <= 25.0:
        agreement_level = 'Partial agreement'
    else:
        agreement_level = 'Large difference'

    direction = 'higher than' if signed_difference >= 0 else 'lower than'
    summary = (
        f'The general model gave {general_probability_percent:.2f}% Aura, which is '
        f'{absolute_difference:.2f} percentage points {direction} the transparent program score of '
        f'{transparent_probability_percent:.2f}%.'
    )
    transparency_note = (
        'The transparent program shows exactly how closed arms, serious face, and glasses contributed to the score. '
        'The general model only gives an Aura probability, so it is harder to know which visual feature caused its decision.'
    )

    return {
        'transparent_program_score': transparent_probability_percent,
        'general_model_aura_probability': general_probability_percent,
        'signed_difference_percentage_points': signed_difference,
        'absolute_difference_percentage_points': absolute_difference,
        'agreement_level': agreement_level,
        'summary': summary,
        'transparency_note': transparency_note
    }

def make_report(results):
    final_score = float(np.clip(sum(item['contribution'] for item in results['models']), 0.0, 100.0))
    grade = grade_from_score(final_score)
    explanation_sentences = [item['explanation'] for item in results['models']]
    general_model_result = results.get('general_model')
    general_comparison = compare_general_model_to_program(final_score, general_model_result)

    report_lines = []
    report_lines.append('AURA SCORE REPORT')
    report_lines.append('Equation:')
    report_lines.append(AURA_EQUATION)
    report_lines.append('')
    report_lines.append(f'Face crop status: {results["face_crop_note"]}')
    report_lines.append('')

    for index, item in enumerate(results['models'], start=1):
        report_lines.append(f'Model {index}: {item["display_name"]}')
        report_lines.append('All class probabilities:')
        for label, probability in item['all_probabilities'].items():
            report_lines.append(f'- {friendly_label_name(label)}: {probability * 100:.2f}%')
        report_lines.append(f'Predicted target probability: {item["target_probability"] * 100:.2f}%')
        report_lines.append(f'Contribution: {item["contribution"]:.2f} / 100')
        report_lines.append(f'Reason: {item["explanation"]}')
        report_lines.append('')

    report_lines.append('Transparent program final score:')
    report_lines.append(f'{final_score:.2f} / 100')
    report_lines.append('Grade:')
    report_lines.append(grade)
    report_lines.append('')

    if general_model_result:
        report_lines.append('General model comparison:')
        report_lines.append('All class probabilities:')
        for label, probability in general_model_result['all_probabilities'].items():
            report_lines.append(f'- {friendly_label_name(label)}: {probability * 100:.2f}%')
        report_lines.append(f'General model Aura probability: {general_model_result["target_probability"] * 100:.2f}%')
        report_lines.append('Reason: This model gives a direct Aura or not Aura decision, but it does not explain which visual feature caused the decision.')
        report_lines.append(f'Comparison: {general_comparison["summary"]}')
        report_lines.append(f'Agreement level: {general_comparison["agreement_level"]}')
        report_lines.append('')
        report_lines.append('Transparency lesson:')
        report_lines.append(general_comparison['transparency_note'])

    json_result = {
        'input_image_path': results['input_image_path'],
        'face_crop_path': results['face_crop_path'],
        'whether_face_crop_was_used': bool(results['whether_face_crop_was_used']),
        'face_crop_note': results['face_crop_note'],
        'equation': AURA_EQUATION,
        'normalization_mode': NORMALIZATION_MODE,
        'models': results['models'],
        'transparent_component_models': results['models'],
        'general_model': general_model_result,
        'general_model_comparison': general_comparison,
        'final_aura_score': final_score,
        'grade': grade,
        'explanation_sentences': explanation_sentences
    }

    return '\n'.join(report_lines), json_result


In [ ]:
# Cell 6: Load the transparent models and the general comparison model once
TARGET_CLASS_NAMES = {
    'closed_arms': ['closed arms', 'closed_arms', 'closed', 'closed hands', 'closed_hands', 'crossed arms', 'crossed_arms', 'crossed', 'crossed hands', 'crossed_hands'],
    'serious_face': ['serious face', 'serious_face', 'serious', 'non_smile', 'not_smile', 'not smile', 'no_smile', 'not smiling', 'neutral'],
    'glasses': ['glasses', 'wearing_glasses', 'with_glasses', 'has_glasses'],
    'general_aura': ['aura', 'has_aura', 'has aura', 'aura_score', 'aura score']
}

NEGATIVE_CLASS_NAMES = {
    'closed_arms': ['open arms', 'open_arms', 'open hands', 'open_hands'],
    'serious_face': ['not serious face', 'not_serious_face', 'not serious', 'not_serious', 'smile', 'smiling'],
    'glasses': ['no_glasses', 'no glasses', 'without_glasses', 'without glasses'],
    'general_aura': ['not_aura', 'not aura', 'no_aura', 'no aura', 'without_aura', 'without aura']
}

MODEL_SPECS = [
    {
        'key': 'closed_arms',
        'display_name': 'Closed arms',
        'model_dir': CROSSED_MODEL_DIR,
        'image_source': 'full_image',
        'reason_subject': 'closed arms'
    },
    {
        'key': 'serious_face',
        'display_name': 'Serious face',
        'model_dir': SMILE_MODEL_DIR,
        'image_source': 'face_crop',
        'reason_subject': 'a serious face'
    },
    {
        'key': 'glasses',
        'display_name': 'Glasses',
        'model_dir': GLASSES_MODEL_DIR,
        'image_source': 'face_crop',
        'reason_subject': 'glasses'
    }
]

GENERAL_MODEL_SPEC = {
    'key': 'general_aura',
    'display_name': 'General black-box Aura model',
    'model_dir': GENERAL_MODEL_DIR,
    'image_source': 'full_image',
    'reason_subject': 'Aura'
}

LOADED_AURA_MODELS = []

for spec in MODEL_SPECS:
    print(f'Loading model once: {spec["display_name"]}')
    model_path = find_model_file(spec['model_dir'])
    labels = read_labels(spec['model_dir'])

    if labels:
        print('Labels:', labels)
    else:
        print('Warning: no labels.txt or metadata.json labels were found. Generic class labels will be used.')

    model = load_keras_model(model_path)
    loaded_item = dict(spec)
    loaded_item.update({
        'model': model,
        'model_path': model_path,
        'labels': labels,
        'input_size': list(get_input_size(model))
    })
    LOADED_AURA_MODELS.append(loaded_item)

    print(f'Done loading {spec["display_name"]}. Input size: {loaded_item["input_size"]}')
    print('')

print('Loading model once: General black-box Aura model')
general_model_path = find_model_file(GENERAL_MODEL_SPEC['model_dir'])
general_labels = read_labels(GENERAL_MODEL_SPEC['model_dir'])
if general_labels:
    print('Labels:', general_labels)
else:
    print('Warning: no labels.txt or metadata.json labels were found for the general model. Generic class labels will be used.')

general_model = load_keras_model(general_model_path)
LOADED_GENERAL_MODEL = dict(GENERAL_MODEL_SPEC)
LOADED_GENERAL_MODEL.update({
    'model': general_model,
    'model_path': general_model_path,
    'labels': general_labels,
    'input_size': list(get_input_size(general_model))
})
print(f'Done loading General black-box Aura model. Input size: {LOADED_GENERAL_MODEL["input_size"]}')
print('')

print('All models are loaded and ready.')
print('Now rerun Cell 7 only whenever you want to check a new image, or run Cell 8 to use the website interface.')


Loading model once: Closed arms
Labels: ['crossed_arms', 'open_arms']
Done loading Closed arms. Input size: [224, 224]

Loading model once: Serious face
Labels: ['non_smile', 'smile']


Done loading Serious face. Input size: [224, 224]

Loading model once: Glasses
Labels: ['glasses', 'no_glasses']
Done loading Glasses. Input size: [224, 224]

Loading model once: General black-box Aura model
Labels: ['aura', 'not_aura']
Done loading General black-box Aura model. Input size: [224, 224]

All models are loaded and ready.
Now rerun Cell 7 only whenever you want to check a new image, or run Cell 8 to use the website interface.


In [ ]:
# Cell 7: Website-only testing note
# This cell intentionally does not upload or analyze images.
# Use Cell 8 to launch the Gradio website and test photos there.

if 'LOADED_AURA_MODELS' not in globals() or not LOADED_AURA_MODELS:
    print('Models are not loaded yet.')
    print('Run Cells 1 to 6 first, then run Cell 8 to open the website interface.')
else:
    print('Models are loaded and ready.')
    print('Run Cell 8 to launch the Aura Score Analyzer website.')
    print('Upload and test images inside the website interface only.')


In [ ]:
# Cell 8: Launch a simple website interface with Gradio
# Run Cells 1-6 once first. Then run this cell to open the website.
import json
import os
import shutil
import gradio as gr
from pathlib import Path

if 'LOADED_AURA_MODELS' not in globals() or not LOADED_AURA_MODELS:
    raise RuntimeError('Models are not loaded yet. Run Cell 6 once before launching the website interface.')

GRADIO_INPUT_DIR = '/content/aura_gradio_inputs'
os.makedirs(GRADIO_INPUT_DIR, exist_ok=True)

CUSTOM_CSS = """
#main-title {
    text-align: center;
    margin-bottom: 8px;
}
#score-box textarea, #grade-box textarea {
    font-size: 22px;
    font-weight: 700;
    text-align: center;
}
.report-box textarea {
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace;
    font-size: 14px;
    line-height: 1.45;
}
"""

def analyze_aura_from_website(uploaded_image_path):
    if uploaded_image_path is None:
        raise gr.Error('Please upload one image first.')

    input_suffix = Path(uploaded_image_path).suffix.lower()
    if input_suffix not in {'.jpg', '.jpeg', '.png', '.webp'}:
        raise gr.Error('Please upload a jpg, jpeg, png, or webp image.')

    stable_input_path = os.path.join(GRADIO_INPUT_DIR, f'current_input{input_suffix}')
    shutil.copy(uploaded_image_path, stable_input_path)

    face_crop_path, face_crop_used, face_crop_message = crop_face(stable_input_path)

    run_results = {
        'input_image_path': stable_input_path,
        'face_crop_path': face_crop_path,
        'whether_face_crop_was_used': face_crop_used,
        'face_crop_note': face_crop_message,
        'models': []
    }

    progress_lines = []
    for item in LOADED_AURA_MODELS:
        image_path = stable_input_path if item['image_source'] == 'full_image' else face_crop_path
        prediction = predict_model(item['model'], image_path, item['labels'])
        target_probability, target_label, target_match_type = get_target_probability(
            prediction,
            TARGET_CLASS_NAMES[item['key']],
            NEGATIVE_CLASS_NAMES.get(item['key'])
        )

        contribution = float(np.clip(target_probability * EQUAL_MODEL_WEIGHT, 0.0, EQUAL_MODEL_WEIGHT))
        explanation = build_reason(item['reason_subject'], target_probability)

        run_results['models'].append({
            'key': item['key'],
            'display_name': item['display_name'],
            'model_dir': item['model_dir'],
            'model_path': item['model_path'],
            'image_used': image_path,
            'input_size': item['input_size'],
            'labels': prediction['labels'],
            'raw_values': prediction['raw_values'],
            'all_probabilities': prediction['probability_map'],
            'target_label': target_label,
            'target_match_type': target_match_type,
            'target_probability': float(target_probability),
            'contribution': contribution,
            'explanation': explanation
        })
        progress_lines.append(f'{item["display_name"]}: {target_probability * 100:.2f}% target probability')

    if 'LOADED_GENERAL_MODEL' in globals() and LOADED_GENERAL_MODEL:
        item = LOADED_GENERAL_MODEL
        prediction = predict_model(item['model'], stable_input_path, item['labels'])
        target_probability, target_label, target_match_type = get_target_probability(
            prediction,
            TARGET_CLASS_NAMES[item['key']],
            NEGATIVE_CLASS_NAMES.get(item['key'])
        )

        run_results['general_model'] = {
            'key': item['key'],
            'display_name': item['display_name'],
            'model_dir': item['model_dir'],
            'model_path': item['model_path'],
            'image_used': stable_input_path,
            'input_size': item['input_size'],
            'labels': prediction['labels'],
            'raw_values': prediction['raw_values'],
            'all_probabilities': prediction['probability_map'],
            'target_label': target_label,
            'target_match_type': target_match_type,
            'target_probability': float(target_probability),
            'explanation': 'The general model gives an Aura probability, but it does not explain which visual feature caused the decision.'
        }
        progress_lines.append(f'General black-box Aura model: {target_probability * 100:.2f}% Aura probability')

    report_text, result_json = make_report(run_results)

    result_json_path = '/content/aura_result.json'
    with open(result_json_path, 'w', encoding='utf-8') as file:
        json.dump(result_json, file, indent=2)

    score_text = f'{result_json["final_aura_score"]:.2f} / 100'
    grade_text = result_json['grade']
    general_text = 'General model was not available.'
    comparison_text = 'No general model comparison was created.'
    if result_json.get('general_model'):
        general_text = f'{result_json["general_model"]["target_probability"] * 100:.2f}% Aura probability'
    if result_json.get('general_model_comparison'):
        comparison = result_json['general_model_comparison']
        comparison_text = comparison['summary'] + '\n' + comparison['transparency_note']
    status_text = face_crop_message + '\n' + '\n'.join(progress_lines) + f'\nSaved JSON: {result_json_path}'

    return stable_input_path, face_crop_path, score_text, grade_text, general_text, comparison_text, report_text, result_json, status_text

with gr.Blocks(title='Aura Score Analyzer', css=CUSTOM_CSS) as demo:
    gr.Markdown('# Aura Score Analyzer', elem_id='main-title')

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(label='Input photo', type='filepath', height=360)
            analyze_button = gr.Button('Analyze Aura', variant='primary')
            status_output = gr.Textbox(label='Status', lines=5)
        with gr.Column(scale=1):
            score_output = gr.Textbox(label='Final score', elem_id='score-box')
            grade_output = gr.Textbox(label='Grade', elem_id='grade-box')
            general_output = gr.Textbox(label='General model result')
            comparison_output = gr.Textbox(label='Transparency comparison', lines=4)

    with gr.Row():
        original_output = gr.Image(label='Original image', type='filepath', height=320)
        face_output = gr.Image(label='Face crop or fallback', type='filepath', height=320)

    report_output = gr.Textbox(label='Aura score report', lines=22, elem_classes=['report-box'])
    json_output = gr.JSON(label='Full JSON result')

    analyze_button.click(
        fn=analyze_aura_from_website,
        inputs=image_input,
        outputs=[
            original_output,
            face_output,
            score_output,
            grade_output,
            general_output,
            comparison_output,
            report_output,
            json_output,
            status_output
        ]
    )

print('Launching the Aura Score Analyzer website...')
print('Use the public Gradio link that appears below.')
demo.launch(share=True, debug=True)


/tmp/ipykernel_19888/1328198407.py:129: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title='Aura Score Analyzer', css=CUSTOM_CSS) as demo:


Launching the Aura Score Analyzer website...
Use the public Gradio link that appears below.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b007f5397ef64fa9a1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
